# Calibration 04-1 — Nelder-Mead Fit (2019-2020, 7 age groups)

**설정**:
- 22D 파라미터 (β 4 + φ 14 + γ_report + amp + base + sigma)
- Gaussian seasonality
- first_peak_only (week ≥ 26 weight=0)
- Auto-seed from HIRA baseline (γ_report_assumed=200.0 (corner 탈출용 100× seed 축소))
- initial_immunity=0.3

**예상 시간**: 60-90분.

출력: `outputs/calibration/2019-2020_by_age_NM_HIRA.json` + 시각화 PNG.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from kt_data import HIRA_AGE_GROUPS
from kt_epimodel_hira.calibration.hira_target import (
    load_hira_target_by_age, simulation_to_hira_by_age,
)
from kt_epimodel_hira.calibration.optimizer import (
    optimize_calibration_by_age, save_result, load_result,
)
from kt_epimodel_hira.calibration.simple_model import (
    build_aggregated_inputs, simulate_aggregated,
    estimate_initial_infected_from_hira,
)
from kt_epimodel_hira.model.parameters import (
    DiseaseParameters, ModelParameters,
)

OUT = Path('../outputs/calibration'); OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. HIRA target 확인

In [ ]:
target_age = load_hira_target_by_age('2019-2020', first_peak_only=True, first_peak_end_week=26)
weeks = target_age['week_in_season']

fig, ax = plt.subplots(figsize=(13, 5))
for ag in HIRA_AGE_GROUPS:
    ax.plot(weeks, target_age['hira_counts'][ag], 'o-', markersize=3, label=ag)
ax.axvspan(26, 52, alpha=0.1, color='gray', label='excluded (first_peak_only)')
ax.set_xlabel('Week in season'); ax.set_ylabel('HIRA episode count')
ax.set_title('2019-2020 HIRA by age group')
ax.legend(loc='upper right', fontsize=9, ncol=2); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Nelder-Mead 본격 fit (max_iter=2000)

Gradient-free, robust. 22D 공간에서 5,000-10,000 evals 가량 소요.

## Nelder-Mead chained refinement (LBFGS warm start)

진단 결과: NM 은 23-D plateau 에서 `initial_guess()` 부근에 갇혀 한 발짝도 못 움직임 (4,700 eval에도 NLL 변동 없음).

**처치**: LBFGS 결과를 NM 의 시작점으로 → local refinement 만 담당.

**목적**

- LBFGS 결과의 robustness 확인 (NM 이 못 움직이면 진짜 local minimum)
- 다른 알고리즘으로 confirmation
- NM 이 더 낮은 NLL 찾으면 LBFGS 가 stuck 됐다는 증거

저장 경로: `outputs/calibration/2019-2020_by_age_NM_chained_HIRA.json` (기존 `_NM_HIRA.json` 과 분리)

예상 시간: 1–2 시간 (max_iter=2000)


In [ ]:
LBFGS_PATH = OUT / '2019-2020_by_age_LBFGS_HIRA.json'
result_lbfgs_init = load_result(LBFGS_PATH)
print(f"Loaded LBFGS warm start:")
print(f"  NLL: {result_lbfgs_init.nll_initial:.2f} → {result_lbfgs_init.nll:.2f}")
cal0 = result_lbfgs_init.calibration
print(f"  β: h={cal0.beta_h:.4f}, w={cal0.beta_w:.4f}, "
      f"s={cal0.beta_s:.4f}, o={cal0.beta_o:.4f}")
print(f"  γ_report={cal0.gamma_report:.4f}, "
      f"amp={result_lbfgs_init.seasonality_amp:.4f}, "
      f"σ={result_lbfgs_init.seasonality_sigma:.2f}, "
      f"peak_day={result_lbfgs_init.seasonality_peak_day:.1f}")

t0 = time.time()
result_nm = optimize_calibration_by_age(
    season='2019-2020',
    use_data_seed=True,
    gamma_report_assumed=0.2,
    seed_e_factor=0.5,
    initial_immunity=0.3,
    method='Nelder-Mead',
    max_iterations=2000,
    first_peak_only=True,
    first_peak_end_week=26,
    t_span=(0.0, 364.0),
    initial_from_result=result_lbfgs_init,  # warm start
    verbose=True,
)
print(f"\nTotal elapsed: {(time.time() - t0) / 60:.1f} min")
save_result(result_nm, OUT / '2019-2020_by_age_NM_chained_HIRA.json')

print(f"\n=== NM chained refinement 결과 ===")
print(f"  LBFGS NLL (start):  {result_lbfgs_init.nll:.2f}")
print(f"  NM NLL (final):     {result_nm.nll:.2f}")
print(f"  개선: {result_lbfgs_init.nll - result_nm.nll:+.2f}")
print(f"  NM evals: {result_nm.n_evaluations}")


## 3. Fit 결과 — 7 그룹 관측 vs 예측

In [ ]:
inputs = build_aggregated_inputs()
pop_15 = inputs['pop_15'].flatten()

disease = DiseaseParameters(
    seasonality_mode=result_nm.seasonality_mode,
    seasonality_amp=result_nm.seasonality_amp,
    seasonality_base=result_nm.seasonality_base,
    seasonality_sigma=result_nm.seasonality_sigma,
    seasonality_peak_day=result_nm.seasonality_peak_day,
)
params = ModelParameters(disease=disease).with_calibration(result_nm.calibration)

if result_nm.use_data_seed:
    seed_by_age = estimate_initial_infected_from_hira(
        '2019-2020', pop_15,
        gamma_report_assumed=result_nm.gamma_report_assumed,
    )
else:
    seed_by_age = None

sim = simulate_aggregated(
    params, inputs,
    seed_total=result_nm.seed_total if not result_nm.use_data_seed else 0.0,
    seed_by_age=seed_by_age,
    initial_immunity=result_nm.initial_immunity,
    t_span=(0.0, 364.0),
)
daily_inc = sim.daily_new_infection_by_age()  # Δ(E+I+R), excludes vax flux
predictions = simulation_to_hira_by_age(
    daily_inc, result_nm.calibration.gamma_report, n_weeks=52,
)

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True)
for i, ag in enumerate(HIRA_AGE_GROUPS):
    ax = axes[i // 4, i % 4]
    ax.plot(weeks, target_age['hira_counts'][ag], 'ko-', markersize=3, label='observed')
    ax.plot(weeks, predictions[ag], 'r-', linewidth=2, label='predicted (NM)')
    ax.axvspan(26, 52, alpha=0.1, color='gray')
    ax.set_title(f'Age {ag}')
    ax.grid(True, alpha=0.3)
    if i % 4 == 0:
        ax.set_ylabel('HIRA count')
    if i // 4 == 1:
        ax.set_xlabel('Week')
axes[1, 3].axis('off')
axes[0, 0].legend(loc='upper right', fontsize=8)
fig.suptitle(
    f'2019-2020 Nelder-Mead fit (mode={result_nm.seasonality_mode}) — '
    f'NLL {result_nm.nll_initial:.0f} → {result_nm.nll:.0f}',
    fontsize=13,
)
plt.tight_layout()
plt.savefig(OUT / '2019-2020_by_age_NM_fit.png', dpi=120)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
AGES = ['0-4','5-9','10-14','15-19','20-24','25-29','30-34','35-39',
        '40-44','45-49','50-54','55-59','60-64','65-69','70+']
axes[0].bar(np.arange(15), result_nm.calibration.phi)
axes[0].axhline(1.0, color='red', linestyle='--', label='reference')
axes[0].set_xticks(np.arange(15)); axes[0].set_xticklabels(AGES, rotation=45)
axes[0].set_ylabel(r'$\phi_a$')
axes[0].set_title('Age-specific susceptibility (NM fit)')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

channels = ['home', 'work', 'school', 'other']
betas = [result_nm.calibration.beta_h, result_nm.calibration.beta_w,
         result_nm.calibration.beta_s, result_nm.calibration.beta_o]
axes[1].bar(channels, betas, color=['C0', 'C1', 'C2', 'C3'])
axes[1].set_ylabel(r'$\beta_{ch}$')
axes[1].set_title(
    f'Channel β (γ_r={result_nm.calibration.gamma_report:.3f}, '
    f'amp={result_nm.seasonality_amp:.2f}, base={result_nm.seasonality_base:.2f}, '
    f'σ={result_nm.seasonality_sigma:.0f})'
)
axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(OUT / '2019-2020_by_age_NM_phi_beta.png', dpi=120)
plt.show()

print(pl.DataFrame([{
    'method': result_nm.method,
    'nll_initial': result_nm.nll_initial,
    'nll_final': result_nm.nll,
    'n_evals': result_nm.n_evaluations,
    'elapsed_min': result_nm.elapsed_seconds / 60,
    'beta_h': result_nm.calibration.beta_h,
    'beta_w': result_nm.calibration.beta_w,
    'beta_s': result_nm.calibration.beta_s,
    'beta_o': result_nm.calibration.beta_o,
    'gamma_report': result_nm.calibration.gamma_report,
    'amp': result_nm.seasonality_amp,
    'base': result_nm.seasonality_base,
    'sigma': result_nm.seasonality_sigma,
    'phi_min': float(result_nm.calibration.phi.min()),
    'phi_max': float(result_nm.calibration.phi.max()),
}]))